In [1]:
import pandas as pd
import re
from sklearn.preprocessing import LabelEncoder
import numpy as np
import geopandas as gpd

In [2]:
df = pd.read_csv('/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/all_results.csv')

In [3]:
# unify the symbols output format
def clean_column_pattern(value, row_idx, col_name):
    if pd.isna(value):
        return value
    value_str = str(value)

    match = re.search(r'([a-zA-Z]+\d+)\)?', value_str)
    if match:
        return match.group(1)
    print(f"Row {row_idx}, Column '{col_name}': Unmatched value '{value_str}'")
    return value

columns_to_clean = [col for col in df.columns if '_Symbol' in col and col != 'Demo_which_state_of_the_usa_do_they_live_in__Symbol']
for col in columns_to_clean:
    df[col] = pd.Series(
        (clean_column_pattern(val, row_idx=idx, col_name = col) for idx, val in df[col].items()),
        index = df.index
    )

In [4]:
# Standardize letter prefixes in symbol columns (sometimes there are a multiple letter prefix in one column)
for col in columns_to_clean:
    non_null_values = df[col].dropna()

    if len(non_null_values) == 0:
        print(f"Column '{col}' has no non-null values.\n")
        continue

    letter_prefixes = set()
    prefix_counts = {}
    for idx, val in non_null_values.items():
        match = re.match(r'([a-zA-Z]+)', str(val))
        if match:
            prefix = match.group(1)
            letter_prefixes.add(prefix)
            prefix_counts[prefix] = prefix_counts.get(prefix, 0) + 1

    if len(letter_prefixes) > 1:
        print(f"Column '{col}' has multiple letter prefixes: {letter_prefixes}")
        for prefix, count in sorted(prefix_counts.items()):
            print(f"  {prefix}: {count}")
        # Find the most common prefix
        most_common_prefix = max(prefix_counts, key=prefix_counts.get)

        df[col] = df[col].apply(
            lambda x: re.sub(r'^[a-zA-Z]+', most_common_prefix, str(x)) if pd.notna(x) else x
        )
        print()

Column 'Belief_opinion_on_the_number_of_immigrants_in_america_Symbol' has multiple letter prefixes: {'imd', 'ima'}
  ima: 10490
  imd: 2

Column 'Belief_opinion_on_the_statement_immigrants_take_jobs_away_from_people_who_were_born_in_the_usa_Symbol' has multiple letter prefixes: {'imc', 'ima', 'imb'}
  ima: 1
  imb: 3
  imc: 10488

Column 'Belief_opinion_on_what_should_be_done_with_illegal_immigrants_Symbol' has multiple letter prefixes: {'imd', 'imc'}
  imc: 2
  imd: 10490

Column 'Belief_opinion_on_the_statement_men_are_better_suited_emotionally_for_politics_than_are_most_women_Symbol' has multiple letter prefixes: {'fea', 'feb'}
  fea: 10486
  feb: 6

Column 'Belief_opinion_on_declining_share_of_white_people_in_america_Symbol' has multiple letter prefixes: {'rab', 'raa'}
  raa: 1
  rab: 10491

Column 'Belief_opinion_on_the_statement_because_of_past_discrimination_black_people_should_be_given_preference_in_hiring_and_promotion_Symbol' has multiple letter prefixes: {'rac', 'fra'}
  fra

In [5]:
# Rename columns by replacing "__" with "_" and print every replacement
old_cols = df.columns.tolist()
new_cols = [c.replace('__', '_') for c in old_cols]

for old, new in zip(old_cols, new_cols):
    if old != new:
        print(f"{old}  ->  {new}")

df.columns = new_cols
print(f"Total renamed columns: {sum(o != n for o, n in zip(old_cols, new_cols))}")

Belief_trust_in_the_information_that_comes_from_independent_media_sources_including_podcasts_blogs_socialmedia_etc__Category  ->  Belief_trust_in_the_information_that_comes_from_independent_media_sources_including_podcasts_blogs_socialmedia_etc_Category
Belief_trust_in_the_information_that_comes_from_independent_media_sources_including_podcasts_blogs_socialmedia_etc__Explanation  ->  Belief_trust_in_the_information_that_comes_from_independent_media_sources_including_podcasts_blogs_socialmedia_etc_Explanation
Belief_trust_in_the_information_that_comes_from_independent_media_sources_including_podcasts_blogs_socialmedia_etc__Symbol  ->  Belief_trust_in_the_information_that_comes_from_independent_media_sources_including_podcasts_blogs_socialmedia_etc_Symbol
Belief_trust_in_the_information_that_comes_from_independent_media_sources_including_podcasts_blogs_socialmedia_etc__Speculation  ->  Belief_trust_in_the_information_that_comes_from_independent_media_sources_including_podcasts_blogs_soci

In [6]:
# df_state = df[['Demo_which_state_of_the_usa_do_they_live_in__Symbol']].copy()
# print(f"Number of rows with state category 'text)': {(df_state['Demo_which_state_of_the_usa_do_they_live_in__Symbol'] == 'text) ny').sum()}")
# # Find rows where state is 'text)'
text_rows = df[df['Demo_which_state_of_the_usa_do_they_live_in_Symbol'] == 'xx']
print(f"Example row with 'text)' state:\n")
print(text_rows[['Demo_which_state_of_the_usa_do_they_live_in_Symbol', 'Demo_which_state_of_the_usa_do_they_live_in_Explanation']].head(1))

# Print explaination for states with weird category lables
if len(text_rows) > 0:
    pd.set_option('display.max_colwidth', None)
    explanation = text_rows['Demo_which_state_of_the_usa_do_they_live_in_Explanation'].iloc[0]
    print("\nFull explanation:")
    print(explanation)

Example row with 'text)' state:

      Demo_which_state_of_the_usa_do_they_live_in_Symbol  \
14395                                                 xx   

      Demo_which_state_of_the_usa_do_they_live_in_Explanation  
14395  the profile states a selfreported location of ...       

Full explanation:
the profile states a selfreported location of paris, philadelphia. paris is in france and philadelphia is a city in the united states, but no u.s. state is mentioned. because the user clearly lists a nonu.s. city (paris) and only a u.s. city without a state, we conclude the user is not tied to any u.s. state. this is a highspeculation inference since the data give no explicit state information.


In [7]:
# Unified state cleanup mapping
STATE_MAPPING = {
    # Standard abbreviations
    'alabama': 'AL', 'alaska': 'AK', 'arizona': 'AZ', 'arkansas': 'AR', 'california': 'CA',
    'colorado': 'CO', 'connecticut': 'CT', 'delaware': 'DE', 'florida': 'FL', 'georgia': 'GA',
    'hawaii': 'HI', 'idaho': 'ID', 'illinois': 'IL', 'indiana': 'IN', 'iowa': 'IA',
    'kansas': 'KS', 'kentucky': 'KY', 'louisiana': 'LA', 'maine': 'ME', 'maryland': 'MD',
    'massachusetts': 'MA', 'michigan': 'MI', 'minnesota': 'MN', 'mississippi': 'MS', 'missouri': 'MO',
    'montana': 'MT', 'nebraska': 'NE', 'nevada': 'NV', 'new hampshire': 'NH', 'new jersey': 'NJ',
    'new mexico': 'NM', 'new york': 'NY', 'north carolina': 'NC', 'north dakota': 'ND', 'ohio': 'OH',
    'oklahoma': 'OK', 'oregon': 'OR', 'pennsylvania': 'PA', 'rhode island': 'RI', 'south carolina': 'SC',
    'south dakota': 'SD', 'tennessee': 'TN', 'texas': 'TX', 'utah': 'UT', 'vermont': 'VT',
    'virginia': 'VA', 'washington': 'WA', 'west virginia': 'WV', 'wisconsin': 'WI', 'wyoming': 'WY',

    # D.C.
    'district of columbia': 'DC', 'washington dc': 'DC', 'dc': 'DC',

    # Comma variations
    'ny,': 'NY', 'wa,': 'WA',
    'california,': 'CA', 'new york,': 'NY', 'texas,': 'TX', 'florida,': 'FL',
    'illinois,': 'IL', 'massachusetts,': 'MA', 'michigan,': 'MI',
    'minnesota,': 'MN', 'arizona,': 'AZ', 'nevada,': 'NV', 'tennessee,': 'TN',
    'south carolina,': 'SC', 'kansas,': 'KS', 'louisiana,': 'LA', 'maine,': 'ME',
    'connecticut,': 'CT', 'utah,': 'UT', 'idaho,': 'ID',

    # Parenthetical variations
    'california (ca)': 'CA', 'arizona (az)': 'AZ', 'kansas (ks)': 'KS',
    'tennessee (tn)': 'TN', 'hi (hawaii)': 'HI', 'ma (massachusetts)': 'MA',
    'ca (california)': 'CA',

    # USA 
    'usa': 'USA', 'usa,': 'USA', 'us':'USA',
    'usa (no specific state)': 'USA',
    'usa (unspecified state)': 'USA',
    'usa (not a specific state)': 'USA',
    'usa (no specific state identified)': 'USA',
    'usa (the user is from the united states but does not specify a state)': 'USA',
    'usa (the user is from the united states but no specific state is identified)': 'USA',
    'if they do not specify a state, but are still from the united states of america, write "usa"': 'USA',

    # No state specified
    'text': 'No_state', 'na': 'No_state', 'n/a': 'No_state', 'xx': 'No_state',
    'not from a state in the usa': 'No_state',
    'not from a state in the usa,': 'No_state',
    'no state': 'No_state',
}

def clean_state(state):
    if pd.isna(state):
        return np.nan
    s = str(state).strip().lower().rstrip(')')
    # Remove 'text)' prefix if present
    if s.startswith('text)'):
        s = s[6:].strip()

    if s in STATE_MAPPING:
        return STATE_MAPPING[s]
    
    if len(s) == 2 and s.upper() in STATE_MAPPING.values():
        return s.upper()    
    
    # If direct mapping fails, try partial matches, longest keys first to avoid false positives
    for key in sorted(STATE_MAPPING.keys(), key=len, reverse=True):
        if key in s:
            return STATE_MAPPING[key]

    print(f"No match found: '{s}'")
    return s.upper()

df['state_cleaned'] = df['Demo_which_state_of_the_usa_do_they_live_in_Symbol'].apply(clean_state)
print(len(df['state_cleaned'].unique()))
print(df['state_cleaned'].value_counts())

51
state_cleaned
No_state    2582
CA           966
NY           761
TX           557
FL           551
USA          511
PA           350
WA           283
VA           254
AZ           219
MA           209
CO           188
GA           187
OH           181
TN           179
NJ           177
MN           169
IL           166
NC           165
MI           163
DC           154
NV           128
MD           127
OR           126
SC           122
WI            95
KS            93
UT            80
IA            80
IN            67
KY            65
LA            58
AL            54
CT            48
AR            42
HI            39
DE            36
MT            32
ID            32
NH            30
MO            25
WY            18
NM            17
MS            16
SD            15
OK            13
RI            11
VT            10
ME             9
NE             8
Name: count, dtype: int64


/tmp/ipykernel_4141/277044136.py:70: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['state_cleaned'] = df['Demo_which_state_of_the_usa_do_they_live_in_Symbol'].apply(clean_state)


In [8]:
# add lickert order detection for belief columns
belief_cols = df.columns[df.columns.str.startswith('Belief_') & df.columns.str.endswith('_Symbol')]

def extract_number(cat):
    match = re.search(r"(\d+)$", str(cat))
    if match:
        return int(match.group(1))
    else:
        return None
    
def detect_likert_order(df, col):
    cats = df[col].dropna().unique()
    ordered_cats = sorted(cats, key=extract_number)
    return ordered_cats

for col in belief_cols:
    likert_order = detect_likert_order(df, col)

    df[col + '_cat'] = pd.Categorical(df[col], categories=likert_order, ordered=True)
    df[col + '_num'] = df[col + '_cat'].cat.codes.replace(-1, pd.NA) + 1

/tmp/ipykernel_4141/3666066593.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + '_cat'] = pd.Categorical(df[col], categories=likert_order, ordered=True)
/tmp/ipykernel_4141/3666066593.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + '_num'] = df[col + '_cat'].cat.codes.replace(-1, pd.NA) + 1
/tmp/ipykernel_4141/3666066593.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns a

In [9]:
new_df = df[(df['state_cleaned'] != 'No_state') & (df['state_cleaned'] != 'USA')]

In [10]:
subset_df = new_df[new_df["id"]==1081532502966239232]
subset_df[[ "tweets_struct","Demo_which_state_of_the_usa_do_they_live_in_Symbol", "Demo_which_state_of_the_usa_do_they_live_in_Explanation"]]
# subset_df["Demo_which_state_of_the_usa_do_they_live_in_Symbol"].value_counts()
subsubset_df = subset_df[subset_df["tweets_struct"] != "[]"]
len(subset_df)

13

In [11]:
subsubset_df[["Demo_age_Symbol"]]

,Demo_age_Symbol
11,a2


In [12]:
subsubset_df[["Demo_which_state_of_the_usa_do_they_live_in_Symbol"]]

,Demo_which_state_of_the_usa_do_they_live_in_Symbol
11,NaN


In [13]:
subsubset_df[["Demo_age_Symbol"]]

,Demo_age_Symbol
11,a2


In [14]:
# Check how many rows this id has in each df
print(len(df[df["id"] == 1081532502966239232]))        # e.g. 20
print(len(new_df[new_df["id"] == 1081532502966239232])) # e.g. fewer

# See which state_cleaned values are being removed for this id
subset = df[df["id"] == 1081532502966239232]
print(subset[["tweets_struct", "state_cleaned"]].value_counts())

26
13
tweets_struct                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [15]:
subset_df = df[df["id"]==1081532502966239232]
subset_df[[ "tweets_struct","Demo_which_state_of_the_usa_do_they_live_in_Symbol", "Demo_which_state_of_the_usa_do_they_live_in_Explanation"]]
# subset_df["Demo_which_state_of_the_usa_do_they_live_in_Symbol"].value_counts()
subsubset_df = subset_df[subset_df["tweets_struct"] != "[]"]
len(subset_df)

26